## Report
```
The program generates a random DNA sequence of 20,000,000 nucleotides and counts the occurrence of each base (A, C, G, T) in parallel on the GPU. Each thread reads one element and performs a global atomicAdd on one of four shared counters. The kernel is launched with four block sizes — 64, 128, 256, and 512 —. All four configurations produced nearly identical execution times of ~14.45–14.57 ms. The performance shows that the bottleneck is atomic contention on the four counters, not the block size.
```

In [1]:
%%writefile parallel_dna_base_frequency_analysis.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 20000000

__global__ void parallel_dna_base_frequency_analysis(const int *seq, unsigned long long *counters, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n) {
        atomicAdd(&counters[seq[idx]], 1ULL);
    }
}

void launch_dna_kernel(const int *h_seq, int block_size)
{
    printf("\nBlock size : %d\n", block_size);

    int *d_seq = NULL;
    unsigned long long *d_counters = NULL;

    cudaMalloc((void **)&d_seq, N * sizeof(int));
    cudaMalloc((void **)&d_counters, 4 * sizeof(unsigned long long));

    cudaMemset(d_counters, 0, 4 * sizeof(unsigned long long));

    cudaMemcpy(d_seq, h_seq, N * sizeof(int), cudaMemcpyHostToDevice);

    int grid_size = (N + block_size - 1) / block_size;
    long long total_threads = (long long)grid_size * block_size;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    parallel_dna_base_frequency_analysis<<<grid_size, block_size>>>(d_seq, d_counters, N);

    cudaEventRecord(stop);

    cudaEventSynchronize(stop);

    float ms = 0.0f;

    cudaEventElapsedTime(&ms, start, stop);

    unsigned long long h_counters[4] = {0};

    cudaMemcpy(h_counters, d_counters, 4 * sizeof(unsigned long long), cudaMemcpyDeviceToHost);

    unsigned long long total = h_counters[0] + h_counters[1] + h_counters[2] + h_counters[3];

    printf("Count of A : %llu  (%.4f%%)\n", h_counters[0], 100.0 * h_counters[0] / N);
    printf("Count of C : %llu  (%.4f%%)\n", h_counters[1], 100.0 * h_counters[1] / N);
    printf("Count of G : %llu  (%.4f%%)\n", h_counters[2], 100.0 * h_counters[2] / N);
    printf("Count of T : %llu  (%.4f%%)\n", h_counters[3], 100.0 * h_counters[3] / N);
    printf("Total count : %llu  (expected %d)\n", total, N);
    printf("Number of blocks : %d\n", grid_size);
    printf("Total threads launched : %lld\n", total_threads);
    printf("Kernel execution time : %.4f ms\n", ms);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(d_seq);
    cudaFree(d_counters);
}

int main(void)
{
    srand((unsigned int)time(NULL));

    int *h_seq = (int *)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        h_seq[i] = rand() % 4;
    }

    int block_sizes[] = {64, 128, 256, 512};
    int num_configs = sizeof(block_sizes) / sizeof(block_sizes[0]);

    for (int i = 0; i < num_configs; i++) {
        launch_dna_kernel(h_seq, block_sizes[i]);
    }

    free(h_seq);

    return 0;
}

Writing parallel_dna_base_frequency_analysis.cu


In [2]:
!nvcc -arch=sm_75 parallel_dna_base_frequency_analysis.cu -o parallel_dna_base_frequency_analysis

In [3]:
!./parallel_dna_base_frequency_analysis


Block size : 64
Count of A : 5001308  (25.0065%)
Count of C : 4996289  (24.9814%)
Count of G : 4999877  (24.9994%)
Count of T : 5002526  (25.0126%)
Total count : 20000000  (expected 20000000)
Number of blocks : 312500
Total threads launched : 20000000
Kernel execution time : 14.5705 ms

Block size : 128
Count of A : 5001308  (25.0065%)
Count of C : 4996289  (24.9814%)
Count of G : 4999877  (24.9994%)
Count of T : 5002526  (25.0126%)
Total count : 20000000  (expected 20000000)
Number of blocks : 156250
Total threads launched : 20000000
Kernel execution time : 14.4548 ms

Block size : 256
Count of A : 5001308  (25.0065%)
Count of C : 4996289  (24.9814%)
Count of G : 4999877  (24.9994%)
Count of T : 5002526  (25.0126%)
Total count : 20000000  (expected 20000000)
Number of blocks : 78125
Total threads launched : 20000000
Kernel execution time : 14.4540 ms

Block size : 512
Count of A : 5001308  (25.0065%)
Count of C : 4996289  (24.9814%)
Count of G : 4999877  (24.9994%)
Count of T : 50025